## Imports

In [ ]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.master("local").appName("BigData").config("spark.ui.showConsoleProgress", "false").getOrCreate()

base_path = "../data/1_bronze/"

df_sellers = spark.read.parquet(f"{base_path}sellers")
df_customers = spark.read.parquet(f"{base_path}customers")
df_geo = spark.read.parquet(f"{base_path}geolocation")

## Types

In [ ]:
df_sellers.show(3)
df_sellers.printSchema()

df_customers.show(3)
df_customers.printSchema()

df_geo.show(3)
df_geo.printSchema()

dfs = {
    "sellers": df_sellers,
    "customers": df_customers,
    "geolocation": df_geo,
}

## Missing values

In [ ]:
from pyspark.sql import functions as F

for name, df in dfs.items():
    print(name.upper())
    for column in df.columns:
        nulls = df.filter(F.col(column).isNull()).count()
        print(f"   Colonne : {column} | Valeurs nulles : {nulls}")
    print()


## Duplicates

In [ ]:
for name, df in dfs.items():
    print(f"{name.upper()} - {df.count() - df.distinct().count()} doublons")



For the geolocation dataframe, we need to keep only distinct zip_codes since it is a join key with two other dataframes.  
Since there are several latitude / longitude combos for a single zip_code, we use the average latitude and average latitude for each zip_code.

In [ ]:
geo_duplicates = df_geo.groupBy("geolocation_zip_code_prefix").count().filter("count > 1")
print("GEOLOCATION - Duplicated zip codes")
geo_duplicates.show(5)


df_geo_unique = df_geo.groupBy("geolocation_zip_code_prefix").agg(
    F.avg("geolocation_lat").alias("geolocation_lat"),
    F.avg("geolocation_lng").alias("geolocation_lng"),
    F.first("geolocation_city").alias("geolocation_city"),
    F.first("geolocation_state").alias("geolocation_state")
)

print("\n\nGEOLOCATION before removing duplicates -", df_geo.count(), "lines")
print("GEOLOCATION after removing duplicates -", df_geo_unique.count(), "lines")
df_geo_unique.show(5)


## Exports

In [ ]:
for name, df in dfs.items():
    df.write.parquet(f"../data/2_silver/{name}", mode="overwrite")
    df.write.mode("overwrite").parquet(f"{base_path}/data/2_silver/{name}")
    print(f"✔ {name} saved to silver")
